In [1]:
!nvidia-smi

Wed Jul 15 11:30:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.0 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline

In [11]:
cuda_source = """
#include <cuda_runtime.h>
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>

__device__ __forceinline__ float warpReduceSum(float val) {
    #pragma unroll
    for (int offset = warpSize / 2; offset > 0; offset /= 2) {
        val += __shfl_down_sync(0xffffffff, val, offset);
    }
    return val;
}

__global__ void coalesced_warp_sgemv_kernel(const float* __restrict__ matd, const float* __restrict__ vecd, float* __restrict__ resd, int M, int N) {
    int bid = blockIdx.x;
    if (bid >= M) return;

    int tid = threadIdx.x;

    float partial_sum = 0.f;
    for (int col = tid; col < N; col += blockDim.x) {
        partial_sum += matd[bid * N + col] * vecd[col];
    }

    float sum = warpReduceSum(partial_sum);
    if (tid == 0) {
        resd[bid] = sum;
    }
}

torch::Tensor gemv_forward(torch::Tensor mat, torch::Tensor vec) {
    auto mat_c = mat.contiguous();
    auto vec_c = vec.contiguous();

    const int M = mat_c.size(0);
    const int N = mat_c.size(1);

    auto res = torch::empty({M}, mat_c.options());

    const int NUM_THREADS = 32;
    dim3 block_size(NUM_THREADS);
    dim3 grid_size(M);

    coalesced_warp_sgemv_kernel<<<grid_size, block_size>>>(
        mat_c.data_ptr<float>(), vec_c.data_ptr<float>(), res.data_ptr<float>(), M, N);

    return res;
}
"""

In [12]:
cpp_source = "torch::Tensor gemv_forward(torch::Tensor mat, torch::Tensor vec);"

In [15]:
module = load_inline(
    name="gemv_coalesced_warp",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["gemv_forward"],
    verbose=True,
    build_directory=build_dir
)

In [16]:
M, N = 4096, 8192
mat = torch.randn(M, N, device="cuda", dtype=torch.float32)
vec = torch.randn(N, device="cuda", dtype=torch.float32)

res = module.gemv_forward(mat, vec)
ref = torch.mv(mat, vec)

print("shape:", tuple(res.shape))
print("max abs diff:", (res - ref).abs().max().item())
print("allclose:", torch.allclose(res, ref, atol=1e-3, rtol=1e-4))

shape: (4096,)
max abs diff: 0.0
allclose: True
